# T.R.A.C.E. Pipeline

In [368]:
import av
import cv2
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple
from scipy.sparse.csgraph import dijkstra
from skimage.morphology import skeletonize
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter1d, map_coordinates
from scipy.sparse import coo_matrix
from search_data import *
from time import perf_counter

In [369]:
class Logger:
    DO_PRINTING = True

    def __init__(self):
        self.function_calls = 0
        self.function_log = {}

    def f_enter(self, function_name):
        if Logger.DO_PRINTING: print(f"Entered Function: {function_name}")
        f_id = self.function_calls
        self.function_calls += 1
        self.function_log[f_id] = { "name": function_name, "start": perf_counter(), "end": 0, "duration": 0 }
        return f_id
        

    def f_exit(self, f_id: int):        
        self.function_log[f_id]["end"] = perf_counter()
        self.function_log[f_id]["duration"] = self.function_log[f_id]["end"] - self.function_log[f_id]["start"]

        if Logger.DO_PRINTING: 
            print(f"Exiting Function: {self.function_log[f_id]["name"]}")
            print(f"\tTime Elapsed: {self.function_log[f_id]["duration"]}")

p_log = Logger()

## Video Handling

In [370]:
# Load frames one by one in grayscale
class VideoLoader:

    def __init__(self, input_video):
        self.input_video = input_video
        self.container = av.open(input_video)

        stream = self.container.streams.video[0]

        self.fps = max(1, int(round(float(stream.average_rate)))) if stream.average_rate is not None else None
        self.frame_timestep = 1 / self.fps
        self.frame_w = stream.width
        self.frame_h = stream.height
        self.total_frames = stream.frames

        self.frames = self.container.decode(video=0)
        self.frame_ind = 0

        self.duration = (float(self.container.duration) / av.time_base
            if self.container.duration is not None
            else None
        )

    def reset(self):
        self.frames = self.container.decode(video=0)
        self.frame_ind = 0

    def get_next_frame(self):
        res = next(self.frames)
        self.frame_ind += 1
        return res.to_ndarray(format='gray8')

    def get_fps(self):
        return self.fps

    def get_frame_timestep(self):
        return self.frame_timestep

    def get_frame_w(self):
        return self.frame_w

    def get_frame_h(self):
        return self.frame_h

    def get_frame_timestamp(self, frame_ind: int):
        return self.frame_timestep * frame_ind

    def get_last_frame_timestamp(self):
        return self.get_frame_timestamp(self.frame_ind-1)

    def get_total_frames(self):
        return self.total_frames
    
    def get_frame_ind(self):
        return self.frame_ind

    def get_duration(self):
        return self.duration

In [371]:
class VideoWriter:
    
    def __init__(self, output_video, fps, width, height):
        self.output_video = output_video
        self.fps = fps
        self.width = width
        self.height = height

        self.container = av.open(output_video, mode='w')
        self.stream = self.container.add_stream('mpeg4', rate=fps)
        self.stream.width = width
        self.stream.height = height
        self.stream.pix_fmt = 'yuv420p' # Alternatively, 'gray' is grayscale, 'monow' is binary white, and 'monob' is binary black.

    def write_frame(self, frame):
        # Create an AVFrame from the RGB frame
        av_frame = av.VideoFrame.from_ndarray(frame, format='rgb24')

        # Encode the frame and write it to the container
        for packet in self.stream.encode(av_frame):
            self.container.mux(packet)

    def close(self):
        # Flush the encoder and write any remaining packets
        for packet in self.stream.encode():
            self.container.mux(packet)
        
        # Close the container
        self.container.close()

## Preprocessing

In [372]:
def find_background(video, n, k):
    """
    Estimate the background of a video using n sampled frames,
    taking every k-th frame.

    Parameters:
        video: VideoLoader
        n: Number of frames to sample
        k: Number of frames to skip between samples

    Returns:
        A grayscale NumPy array representing the estimated background.
    """
    f_id = p_log.f_enter("find_background")
    video.reset()
    frames = []

    for _ in range(n):
        # Get the next frame
        frame = video.get_next_frame()
        # Finish if the end of the video is reached
        if frame is None: break
        # Append the sample if appropriate
        if n % k == 0: frames.append(frame)

    # Stack frames and calculate median for every pixel
    frames = np.stack(frames, axis=0)
    background = np.median(frames, axis=0).astype(np.uint8)
    
    video.reset()
    p_log.f_exit(f_id)
    return background

In [ ]:
# def apply_threshold(diff: np.ndarray, low: int, high: int, kernel_size: int, min_area: int):
#     # Find each pixel within the proper value range
#     all_candidates = (diff >= low).astype(np.uint8)
#     strong_candidates = (diff >= high).astype(np.uint8)

#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
#     all_candidates = cv2.morphologyEx(all_candidates, cv2.MORPH_CLOSE, kernel)

#     # Find connected regions with at least one strong candidate pixel within it
#     num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(all_candidates, connectivity=8)
#     strong_labels = np.unique(labels[strong_candidates > 0])

#     # Keep only those connected regions
#     keep = np.zeros(num_labels, dtype=bool)
#     keep[strong_labels] = True
#     keep[0] = False

#     # Filter the remaining connected regions by minimum area
#     keep &= stats[:, cv2.CC_STAT_AREA] >= min_area

#     return { "mask" : (keep[labels] * 255).astype(np.uint8), "num_components" : num_labels, "labels" : labels, "stats": stats, "centroids": centroids }
    

# def preprocess_frame(
#     frame: np.ndarray,
#     background: np.ndarray,
#     low: int = 3,
#     high: int = 8,
#     kernel_size: int = 5
# ) -> np.ndarray:
#     f_id = p_log.f_enter("preprocess_frame")
#     # Remove the background from the frame
#     diff = cv2.subtract(background, frame)
   
#     # Create a masked image of the worms, along with connected component stats
#     frame_dat = apply_threshold(diff, low, high, kernel_size, 30)
#     p_log.f_exit(f_id)
#     return frame_dat

import numpy as np, cv2
from skimage.filters import sato

def robust_sigma(x):
    return 1.4826 * np.median(np.abs(x - np.median(x))) + 1e-6

def remove_offset(d):
    small = cv2.resize(d, None, fx=0.1, fy=0.1, interpolation=cv2.INTER_AREA)
    small = cv2.medianBlur(small, 5)
    small = cv2.GaussianBlur(small, (0, 0), 2)
    off = cv2.resize(small, (d.shape[1], d.shape[0]), interpolation=cv2.INTER_LINEAR)
    return d - off

def compute_scores(frame, background, blur_sigma=1.0):
    fr_raw = frame.astype(np.float32)
    bg_raw = background.astype(np.float32)
    d_raw = remove_offset(bg_raw - fr_raw)            # unblurred, gray levels
    fr = cv2.GaussianBlur(fr_raw, (0, 0), blur_sigma)
    bg = cv2.GaussianBlur(bg_raw, (0, 0), blur_sigma)
    d = remove_offset(bg - fr)
    d_z = d / robust_sigma(d)
    r = sato(d, sigmas=(1.0, 1.5), black_ridges=False).astype(np.float32)
    r_z = r / robust_sigma(r)
    return d_z, r_z, d_raw

def apply_threshold(d_z, r_z, low=3.0, high=7.0, ridge_low=5.0,
                    regrow_low=2.5, regrow_radius=4, close_ksize=3,
                    min_length=35, max_area=5000):
    # 1) Hysteresis: weak = plain OR ridge evidence, seeds = plain only
    weak = ((d_z >= low) | (r_z >= ridge_low)).astype(np.uint8)
    strong = d_z >= high
    n, labels = cv2.connectedComponents(weak, connectivity=8)
    keep = np.zeros(n, bool)
    keep[np.unique(labels[strong])] = True
    keep[0] = False
    mask = keep[labels].astype(np.uint8)

    # 2) Constrained regrow: weaker threshold, but only near accepted worms
    k_near = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (2 * regrow_radius + 1, 2 * regrow_radius + 1))
    near = cv2.dilate(mask, k_near) > 0
    mask = (mask.astype(bool) | ((d_z >= regrow_low) & near)).astype(np.uint8)

    # 3) Close gaps (safe now: only worm pixels are in the mask)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_ksize, close_ksize))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    # after step 3 (close), before step 4 (filter):
    tight = (d_z >= tight_low).astype(np.uint8)          # tight_low ~ 3.0-3.5
    k_dil = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    tight = cv2.dilate(tight, k_dil)                      # 1 px of slack
    mask = mask & tight

    # 4) Filter by extent (length), not area, plus an upper area guard
    n, labels, stats, cents = cv2.connectedComponentsWithStats(mask, connectivity=8)
    w, h = stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT]
    area = stats[:, cv2.CC_STAT_AREA]
    ok = (np.hypot(w, h) >= min_length) & (area <= max_area)
    ok[0] = False
    return {"mask": (ok[labels] * 255).astype(np.uint8), "num_components": n,
            "labels": labels, "stats": stats, "centroids": cents}

def preprocess_frame(frame, background, **kw):
    d_z, r_z = compute_scores(frame, background)
    return apply_threshold(d_z, r_z, **kw)

## Single Frame Analysis

In [374]:
def skeletal_analysis(
    mask,
    pixel_size=1.0,
    head_xy=None,
    head_frac=0.1,
    mid_frac=0.5,
    tail_frac=0.9,
    window=0.03,
    smooth_sigma=3.0,
    extend_ends=True,
    return_centerline=False,
):
    f_id = p_log.f_enter("skeletal_analysis")
    """
    Measure length, geodesic midpoint, curvature and width of one worm.
 
    Generated by Claude (Anthropic) -- see the disclaimer at the top of this file.
 
    Method
    ------
    1. Clean the mask (largest component, holes filled).
    2. Skeletonize it and take the longest geodesic path through the skeleton; this
       is the worm's centerline. Side spurs are ignored automatically.
    3. Extend the centerline to the worm's tips, resample it to ~1 px spacing and
       smooth it.
    4. Read length, midpoint, curvature and width off the smoothed centerline.
 
    Parameters
    ----------
    mask : 2D array
        Binary mask containing a single worm (non-zero = worm).
    pixel_size : float
        Physical size of one pixel (e.g. um/px). Length and width are multiplied by
        it; curvature (1/length) is divided by it. Use 1.0 to stay in pixels.
    head_xy : (x, y) or None
        Approximate head location in image coordinates. A mask alone contains no
        head/tail information. If None, the end with the wider cross-section is
        assumed to be the head, which is only a guess -- pass this when you can.
    head_frac, mid_frac, tail_frac : float
        Positions along the body (0 = head tip, 1 = tail tip) where curvature and
        width are sampled. Tips are noisy, so head/tail default to 10% and 90%.
    window : float
        Half-width, as a fraction of body length, averaged around each position.
    smooth_sigma : float
        Gaussian smoothing (pixels) applied to the centerline. Larger values give
        smoother but flatter curvature. Keep it consistent across a dataset.
    extend_ends : bool
        Extend the skeleton to the tips (skeletons stop about one radius short).
    return_centerline : bool
        Also return the smoothed centerline as an (N, 2) array of (x, y) pixels,
        handy for overlaying on the image to check the fit.
 
    Returns
    -------
    dict
        length             : centerline length (pixels * pixel_size)
        geodesic_midpoint  : (x, y) in pixels, halfway along the centerline
        head_point         : (x, y) in pixels, the extreme tip at the head end
        tail_point         : (x, y) in pixels, the extreme tip at the tail end
        curvature          : {"head", "midbody", "tail"}, signed, in 1/length unit
        width              : {"head", "midbody", "tail"}, in length unit
        centerline         : only if return_centerline=True
 
    Limitations
    -----------
    Assumes one worm that does not cross or touch itself. Loops or junctions in the
    skeleton make the longest-path centerline (and therefore length) unreliable.
    Filling holes also closes real gaps inside tightly coiled worms.
    """
 
    # ------------------------------------------------------------------
    # 1. Clean the mask: keep the largest component and fill holes
    # ------------------------------------------------------------------
    mask = np.asarray(mask) > 0
    labels, n_components = ndi.label(mask, structure=np.ones((3, 3)))
    if n_components == 0:
        raise ValueError("Mask is empty.")
    if n_components > 1:
        sizes = ndi.sum(mask, labels, range(1, n_components + 1))
        mask = labels == (np.argmax(sizes) + 1)
    mask = ndi.binary_fill_holes(mask)
 
    # Distance to background, used later to measure width
    dist = ndi.distance_transform_edt(mask)
 
    # ------------------------------------------------------------------
    # 2. Centerline: longest geodesic path through the skeleton
    # ------------------------------------------------------------------
    skeleton = skeletonize(mask)
    pixels = np.argwhere(skeleton)  # (row, col) of every skeleton pixel
    if len(pixels) < 2:
        raise ValueError("Skeleton has fewer than 2 pixels.")
 
    # Graph: one node per skeleton pixel, edges between 8-connected neighbours
    # (weight 1 for side-by-side, sqrt(2) for diagonal).
    node_at = -np.ones(skeleton.shape, dtype=np.int64)
    node_at[pixels[:, 0], pixels[:, 1]] = np.arange(len(pixels))
 
    edge_from, edge_to, edge_weight = [], [], []
    for d_row, d_col in [(0, 1), (1, -1), (1, 0), (1, 1)]:  # forward neighbours only
        nb_row, nb_col = pixels[:, 0] + d_row, pixels[:, 1] + d_col
        inside = ((nb_row >= 0) & (nb_row < skeleton.shape[0]) &
                  (nb_col >= 0) & (nb_col < skeleton.shape[1]))
        src = np.nonzero(inside)[0]
        dst = node_at[nb_row[src], nb_col[src]]
        is_skeleton = dst >= 0
        edge_from.append(src[is_skeleton])
        edge_to.append(dst[is_skeleton])
        edge_weight.append(np.full(is_skeleton.sum(), np.hypot(d_row, d_col)))
 
    graph = coo_matrix(
        (np.concatenate(edge_weight), (np.concatenate(edge_from), np.concatenate(edge_to))),
        shape=(len(pixels), len(pixels)),
    ).tocsr()
 
    # Two sweeps find the geodesically farthest pair of pixels: the pixel farthest
    # from an arbitrary start is one end; the pixel farthest from that is the other.
    dist_from_start = dijkstra(graph, directed=False, indices=0)
    dist_from_start[np.isinf(dist_from_start)] = -1  # ignore disconnected fragments
    end_a = int(np.argmax(dist_from_start))
 
    dist_from_a, predecessors = dijkstra(
        graph, directed=False, indices=end_a, return_predecessors=True
    )
    dist_from_a[np.isinf(dist_from_a)] = -1
    end_b = int(np.argmax(dist_from_a))
 
    # Walk back from end_b to end_a to recover the ordered path
    node_path = [end_b]
    while node_path[-1] != end_a:
        node_path.append(predecessors[node_path[-1]])
    path = pixels[node_path][:, ::-1].astype(float)  # convert to (x, y)
 
    # ------------------------------------------------------------------
    # 3. Extend to the tips, resample to ~1 px spacing, smooth
    # ------------------------------------------------------------------
    if extend_ends and len(path) > 3:
        k = min(10, len(path) - 1)  # look k pixels inward to get the end's direction
        tips = []
        for tip, inward in ((path[0], path[k]), (path[-1], path[-1 - k])):
            direction = tip - inward
            direction /= np.hypot(*direction)
            point = tip.copy()
            for _ in range(400):  # walk outward in half-pixel steps until we leave the mask
                candidate = point + 0.5 * direction
                col, row = int(round(candidate[0])), int(round(candidate[1]))
                inside = 0 <= row < mask.shape[0] and 0 <= col < mask.shape[1]
                if not inside or not mask[row, col]:
                    break
                point = candidate
            tips.append(point)
        path = np.vstack([tips[0], path, tips[1]])
 
    # Remember the extreme end points now: smoothing pulls the ends slightly inward.
    # (With extend_ends=True these sit on the mask boundary at the worm's tips.)
    end_point_1, end_point_2 = path[0].copy(), path[-1].copy()
 
    arc = np.r_[0, np.cumsum(np.hypot(*np.diff(path, axis=0).T))]
    arc_uniform = np.r_[np.arange(0, arc[-1], 1.0), arc[-1]]
    x = gaussian_filter1d(np.interp(arc_uniform, arc, path[:, 0]), smooth_sigma, mode="nearest")
    y = gaussian_filter1d(np.interp(arc_uniform, arc, path[:, 1]), smooth_sigma, mode="nearest")
 
    arc = np.r_[0, np.cumsum(np.hypot(np.diff(x), np.diff(y)))]  # arc length of smoothed curve
    length = arc[-1]
 
    # ------------------------------------------------------------------
    # 4. Width along the centerline
    #    Twice the distance to the background; -1 corrects for pixel-centre sampling.
    # ------------------------------------------------------------------
    width = np.clip(2 * map_coordinates(dist, [y, x], order=1) - 1, 1, None)
 
    # ------------------------------------------------------------------
    # 5. Orient the centerline head -> tail
    # ------------------------------------------------------------------
    if head_xy is not None:
        head_x, head_y = head_xy
        start_is_tail = (np.hypot(x[0] - head_x, y[0] - head_y) >
                         np.hypot(x[-1] - head_x, y[-1] - head_y))
    else:
        n_end = max(int(0.1 * len(arc)), 1)
        start_is_tail = width[:n_end].mean() < width[-n_end:].mean()  # wider end = head
    if start_is_tail:
        x, y, width = x[::-1], y[::-1], width[::-1]
        head_point, tail_point = end_point_2, end_point_1
    else:
        head_point, tail_point = end_point_1, end_point_2
 
    # ------------------------------------------------------------------
    # 6. Signed curvature with respect to arc length:
    #    kappa = (x'y'' - y'x'') / (x'^2 + y'^2)^(3/2)
    # ------------------------------------------------------------------
    dx, dy = np.gradient(x, arc), np.gradient(y, arc)
    ddx, ddy = np.gradient(dx, arc), np.gradient(dy, arc)
    curvature = (dx * ddy - dy * ddx) / np.power(dx**2 + dy**2, 1.5)
 
    # ------------------------------------------------------------------
    # 7. Sample at head / midbody / tail and assemble results
    # ------------------------------------------------------------------
    position = arc / length  # 0 = head tip, 1 = tail tip
 
    def average_near(values, fraction):
        """Mean of `values` within +/- `window` of a normalised body position."""
        near = np.abs(position - fraction) <= window
        if not near.any():
            return float(np.interp(fraction, position, values))
        return float(values[near].mean())
 
    midpoint = (float(np.interp(0.5, position, x)), float(np.interp(0.5, position, y)))
 
    results = {
        "length": length * pixel_size,
        "geodesic_midpoint": midpoint,
        "head_point": (float(head_point[0]), float(head_point[1])),
        "tail_point": (float(tail_point[0]), float(tail_point[1])),
        "curvature": {
            "head": average_near(curvature, head_frac) / pixel_size,
            "midbody": average_near(curvature, mid_frac) / pixel_size,
            "tail": average_near(curvature, tail_frac) / pixel_size,
        },
        "width": {
            "head": average_near(width, head_frac) * pixel_size,
            "midbody": average_near(width, mid_frac) * pixel_size,
            "tail": average_near(width, tail_frac) * pixel_size,
        },
    }
    if return_centerline:
        results["centerline"] = np.column_stack([x, y])

    p_log.f_exit(f_id)
    return results


In [375]:
def analyze_frame(frame_dat, timestamp: float) -> Tuple[np.ndarray, int, List[Dict]]:
    f_id = p_log.f_enter("analyze_frame")
    components = []
    bounding_boxes = []

    for i in range(1, frame_dat["num_components"]):  # Skip the background (label 0)
        # Morphology stats that we care about

        component_mask = (frame_dat["labels"] == i).astype(np.uint8) * 255
        component_skeletal_metrics = skeletal_analysis(component_mask)

        contours, _ = cv2.findContours(frame_dat["mask"], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        largest_contour = max(contours, key=cv2.contourArea)
        hull = cv2.convexHull(largest_contour)
        perimeter = cv2.arcLength(largest_contour, True)

        components.append({
            "timestamp": timestamp,
            "length": component_skeletal_metrics["length"],
            "area": frame_dat["stats"][i, cv2.CC_STAT_AREA],
            "perimeter": perimeter,
            "solidity" : frame_dat["stats"][i, cv2.CC_STAT_AREA] / cv2.contourArea(hull),
            "compactness" : (4 * np.pi * frame_dat["stats"][i, cv2.CC_STAT_AREA]) / (perimeter ** 2),
            "width_head_base" : component_skeletal_metrics["width"]["head"],
            "width_midbody" : component_skeletal_metrics["width"]["midbody"],
            "width_tail_base" : component_skeletal_metrics["width"]["tail"],
            "speed" : 0,                                # Calculated from displacement later
            "motion_mode" : 0,                          # TODO Not sure how to do this yet
            "curvature_head" : component_skeletal_metrics["curvature"]["head"],
            "curvature_midbody" : component_skeletal_metrics["curvature"]["midbody"],
            "d_curvature_midbody" : 0,                  # Calculated later
            "curvature_tail" : component_skeletal_metrics["curvature"]["tail"],
            "coord_x_body" : frame_dat["centroids"][i][0],
            "coord_y_body" : frame_dat["centroids"][i][1]
        })

        bounding_boxes.append({
            "timestamp": timestamp,
            "x": frame_dat["stats"][i, cv2.CC_STAT_LEFT],
            "y": frame_dat["stats"][i, cv2.CC_STAT_TOP],
            "width": frame_dat["stats"][i, cv2.CC_STAT_WIDTH],
            "height": frame_dat["stats"][i, cv2.CC_STAT_HEIGHT],
        })
        
    p_log.f_close(f_id)
    return components, bounding_boxes

## Inter-frame Analysis

In [376]:
# FOR NOW: I need to calculate the delta values for: displacement (which gives speed) and midbody curvature
def delta_calculation(worm_count: int, worm_data: Tuple[List[Dict[str, any]]], frame_timestep: int):
    for i in range(0, worm_count):
        current_worm = worm_data[i]

        # Skip delta calculations for the first row: they will be zero by default, since there is no previous frame to compare to.
        for j in range(1, len(current_worm)):
            current_worm_frame_data = current_worm[j]
            previous_worm_frame_data = current_worm[j-1]

            # Calculate delta values
            dx = current_worm_frame_data["midpoint_x"] - previous_worm_frame_data["midpoint_x"]
            dy = current_worm_frame_data["midpoint_y"] - previous_worm_frame_data["midpoint_y"]
            current_worm["speed"] = np.sqrt(dx**2 + dy**2) / frame_timestep
            current_worm["d_midbody_curvature"] = current_worm_frame_data["midbody_curvature"] - previous_worm_frame_data["midbody_curvature"]

In [377]:
def thrashing_frequency_from_curvature():
    pass

def thrashing_frequency_from_covariance():
    pass

In [378]:
def state_classification():
    pass

In [379]:
from scipy.optimize import linear_sum_assignment

# !!!!!!!!!!!!!!! THIS SHIT IS FROM CLAUDE

FEATS = ["length", "area", "width_midbody"]
BIG = 1e6

CFG = dict(
    base_gate=40.0,        # px, min search radius
    max_speed=60.0,        # px/s, upper bound on plausible worm speed (tune!)
    morph_tol=0.25,        # ~25% log-ratio difference = "1 unit" of morph cost
    w_space=1.0, w_morph=1.0,
    ambig_margin=0.3,      # flag if runner-up cost is within this of chosen cost
    merge_area_ratio=1.4,  # blob this much bigger than a track => likely merge
    max_missed=30,         # frames before a track is closed
    ema=0.1,               # feature smoothing
)

class Track:
    def __init__(self, tid, det):
        self.id = tid
        self.pos = np.array([det["coord_x_body"], det["coord_y_body"]], float)
        self.vel = np.zeros(2)
        self.t = det["timestamp"]
        self.feat = {k: float(det[k]) for k in FEATS}
        self.missed = 0
        self.occluded = False
        self.uncertain = False   # identity may have been swapped in a collision

    def predict(self, t):
        return self.pos + self.vel * (t - self.t)

    def gate(self, t):
        dt = max(t - self.t, 1e-6)
        return max(CFG["base_gate"], CFG["max_speed"] * dt)

    def update(self, det):
        t = det["timestamp"]
        new = np.array([det["coord_x_body"], det["coord_y_body"]], float)
        dt = max(t - self.t, 1e-6)
        if not self.occluded:
            self.vel = 0.5 * self.vel + 0.5 * (new - self.pos) / dt
            for k in FEATS:  # don't pollute features with merged-blob data
                self.feat[k] += CFG["ema"] * (det[k] - self.feat[k])
        self.pos, self.t, self.missed = new, t, 0


def morph_cost(track, det):
    d = [abs(np.log(max(det[k], 1e-6) / max(track.feat[k], 1e-6))) for k in FEATS]
    return np.mean(d) / CFG["morph_tol"]


def build_cost(tracks, dets, t):
    C = np.full((len(tracks), len(dets)), BIG)
    for i, tr in enumerate(tracks):
        pred, gate = tr.predict(t), tr.gate(t)
        for j, d in enumerate(dets):
            dist = np.hypot(d["coord_x_body"] - pred[0], d["coord_y_body"] - pred[1])
            if dist <= gate:
                C[i, j] = CFG["w_space"] * dist / gate + CFG["w_morph"] * morph_cost(tr, d)
    return C


class WormTracker:
    def __init__(self):
        self.tracks, self.next_id, self.closed = [], 0, []

    def step(self, dets, n_expected=None):
        """dets: list of per-component dicts for ONE frame.
        Adds 'track_id', 'flags', 'needs_review' to each det and returns dets."""
        if not dets:
            return dets
        t = dets[0]["timestamp"]
        for d in dets:
            d["flags"] = []
            d["track_id"] = None
        frame_flags = []
        if n_expected is not None and len(dets) != n_expected:
            frame_flags.append(f"count_mismatch({len(dets)}!={n_expected})")

        C = build_cost(self.tracks, dets, t) if self.tracks else np.empty((0, len(dets)))
        matches = {}
        if C.size:
            rows, cols = linear_sum_assignment(C)
            matches = {i: j for i, j in zip(rows, cols) if C[i, j] < BIG}

        # ---- ambiguity check on chosen pairs ----
        for i, j in matches.items():
            others = [C[i, jj] for jj in range(C.shape[1]) if jj != j]
            others += [C[ii, j] for ii in range(C.shape[0]) if ii != i]
            alt = min([c for c in others if c < BIG], default=None)
            if alt is not None and alt - C[i, j] < CFG["ambig_margin"]:
                dets[j]["flags"].append("ambiguous_assignment")

        matched_dets = set(matches.values())
        unmatched_tracks = [i for i in range(len(self.tracks)) if i not in matches]
        unmatched_dets = [j for j in range(len(dets)) if j not in matched_dets]

        # ---- merge / collision detection ----
        merged_dets = {}   # det idx -> list of track idx hidden inside it
        for i in unmatched_tracks:
            tr = self.tracks[i]
            pred, gate = tr.predict(t), tr.gate(t)
            for j, d in enumerate(dets):
                dist = np.hypot(d["coord_x_body"] - pred[0], d["coord_y_body"] - pred[1])
                if dist <= gate and d["area"] > CFG["merge_area_ratio"] * tr.feat["area"]:
                    merged_dets.setdefault(j, []).append(i)
                    break

        # also catch the case where one worm matched a blob that contains it
        # plus another worm (blob much bigger than its track)
        for i, j in list(matches.items()):
            if dets[j]["area"] > CFG["merge_area_ratio"] * self.tracks[i]["feat"]["area"] if False else \
               dets[j]["area"] > CFG["merge_area_ratio"] * self.tracks[i].feat["area"]:
                merged_dets.setdefault(j, []).append(i)
                del matches[i]

        for j, idxs in merged_dets.items():
            ids = [self.tracks[i].id for i in idxs]
            dets[j]["flags"].append(f"merged_blob(tracks={ids})")
            dets[j]["track_id"] = None
            dets[j]["merged_track_ids"] = ids
            for i in idxs:
                tr = self.tracks[i]
                tr.occluded, tr.uncertain = True, True
                tr.missed += 1
                tr.pos, tr.t = tr.predict(t), t   # coast on velocity

        # ---- apply matches ----
        for i, j in matches.items():
            tr = self.tracks[i]
            if tr.uncertain:
                dets[j]["flags"].append("post_collision_identity_uncertain")
                tr.uncertain = False
            tr.occluded = False
            tr.update(dets[j])
            dets[j]["track_id"] = tr.id

        # ---- tracks with nothing at all: lost ----
        still_lost = [i for i in unmatched_tracks
                      if not any(i in v for v in merged_dets.values())]
        for i in still_lost:
            self.tracks[i].missed += 1
            frame_flags.append(f"track_{self.tracks[i].id}_missing")
        survivors = []
        for tr in self.tracks:
            if tr.missed > CFG["max_missed"]:
                self.closed.append(tr)
                frame_flags.append(f"track_{tr.id}_closed")
            else:
                survivors.append(tr)
        self.tracks = survivors

        # ---- new tracks ----
        for j in unmatched_dets:
            if j in merged_dets:
                continue
            tr = Track(self.next_id, dets[j]); self.next_id += 1
            self.tracks.append(tr)
            dets[j]["track_id"] = tr.id
            if len(self.closed) or self.next_id > 1 and t != dets[0]["timestamp"]:
                pass
            dets[j]["flags"].append("new_track")

        for d in dets:
            d["frame_flags"] = frame_flags
            d["needs_review"] = bool(d["flags"] or frame_flags)
        return dets

## Pipeline Class

In [380]:
def make_track_csv(input_csv, column, output_csv=None):
    """
    >> Generated by Claude <<
    Build a wide CSV from `input_csv` where:
      - the first column is 'timestamp'
      - every other column is a unique track_id
      - each cell holds the value of `column` (the label of a column in the
        original CSV) for that track_id at that timestamp (blank if the track
        has no row at that timestamp)

    Rows with a missing track_id are dropped, since they can't be assigned to a
    track column. Returns the resulting DataFrame.
    """
    df = pd.read_csv(input_csv)
    if column not in df.columns:
        raise ValueError(f"'{column}' not found. Available: {list(df.columns)}")

    df = df.dropna(subset=['track_id'])
    df['track_id'] = df['track_id'].astype(int)

    # Fail loudly if a track has two rows at the same timestamp
    if df.duplicated(['timestamp', 'track_id']).any():
        raise ValueError("Duplicate (timestamp, track_id) pairs found.")

    wide = df.pivot(index='timestamp', columns='track_id', values=column)
    wide = wide.sort_index(axis=0).sort_index(axis=1)
    wide.columns.name = None
    wide = wide.reset_index()  # 'timestamp' becomes the first column

    if output_csv is None:
        output_csv = f"{column}_by_track.csv"
    wide.to_csv(output_csv, index=False)
    return wide

In [381]:
# It is assumed that the first frame of the video contains all worms, and that they are clearly seperated.
class TracePipeline:

    def __init__(self, input_video, output_dir):
        self.video_loader = VideoLoader(input_video)
        self.video_writer = VideoWriter(f"{output_dir}/annotated_video.avi", int(self.video_loader.get_fps()), self.video_loader.get_frame_w(), self.video_loader.get_frame_h())
        
        self.background = find_background(self.video_loader, n=100, k=20)

        self.frame_data = []
        self.frames = []
        self.bounding_boxes = []
        self.true_worm_count = 0

        self.tracking_data = []

    def analyze_all_frames(self):
        f_id = p_log.f_enter("TracePipeline::analyze_all_frames")

        if (self.background is None):
            print("Error: Could not find background.")
            return

        # Setup worm identification using the first frame to determine the number of worms and their initial positions
        first_frame = self.video_loader.get_next_frame()

        print(f"\tAnalyzing Frame: {self.video_loader.get_frame_ind()-1} @ {self.video_loader.get_last_frame_timestamp()}")
        
        frame_dat = preprocess_frame(first_frame, self.background)
        worm_dat, bounding_boxes = analyze_frame(frame_dat, self.video_loader.get_last_frame_timestamp())

        self.frame_data.append(worm_dat)
        self.true_worm_count = len(worm_dat)
        self.frames.append(cv2.cvtColor(frame_dat["mask"], cv2.COLOR_GRAY2RGB))
        self.bounding_boxes.append(bounding_boxes)

        # Analyze the video frame by frame
        while True:
            try:
                # Get the next frame from the video
                frame = self.video_loader.get_next_frame()

                print(f"\tAnalyzing Frame: {self.video_loader.get_frame_ind()-1} @ {self.video_loader.get_last_frame_timestamp()}")

                # Preprocess the frame
                frame_dat = preprocess_frame(frame, self.background)

                # Analyze a single frame
                worm_dat, bounding_boxes = analyze_frame(frame_dat, self.video_loader.get_last_frame_timestamp())
                self.frames.append(cv2.cvtColor(frame_dat["mask"], cv2.COLOR_GRAY2RGB))
                self.frame_data.append(worm_dat)
                self.bounding_boxes.append(bounding_boxes)

                # TEMP: So I don't have to wait 20 hours for the test output
                if (self.video_loader.get_frame_ind() > 20): break

            except StopIteration:
                break
        p_log.f_exit(f_id)

    def identify_worms(self):
        f_id = p_log.f_enter("TracePipeline::identify_worms")
        tracker = WormTracker()
        for frame_dat in self.frame_data:
            self.tracking_data.extend(tracker.step(frame_dat, self.true_worm_count))
        p_log.f_exit(f_id)

    def save_outputs(self):
        # Create output video with labelled worms and bounding boxes
        f_id = p_log.f_enter("TracePipeline::save_outputs")
        for i, frame in enumerate(self.frames):

            for j in range(len(self.frame_data[i])):
                worm = self.frame_data[i][j]
                bbox = self.bounding_boxes[i][j]
                track_id = worm["track_id"]
                if track_id is not None:
                    centroid_x = int(worm["coord_x_body"])
                    centroid_y = int(worm["coord_y_body"])
                    cv2.putText(frame, f"ID: {track_id}", (centroid_x + 25, centroid_y - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
                    cv2.circle(frame, (centroid_x, centroid_y), 5, (0, 255, 0), -1)
                    cv2.rectangle(frame, (bbox["x"], bbox["y"]), (bbox["x"] + bbox["width"], bbox["y"] + bbox["height"]), (255, 0, 0), 2)


            self.video_writer.write_frame(frame)
        self.video_writer.close()

        print("Saving Results to output.csv...")
        dataframe = pd.DataFrame(self.tracking_data)
        dataframe.to_csv(f"{OUTPUT_DIR}/output.csv", index=False)
        make_track_csv(f"{OUTPUT_DIR}/output.csv", "length", f"{OUTPUT_DIR}/length_by_track.csv")
        make_track_csv(f"{OUTPUT_DIR}/output.csv", "area", f"{OUTPUT_DIR}/area_by_track.csv")
        make_track_csv(f"{OUTPUT_DIR}/output.csv", "d_curvature_midbody", f"{OUTPUT_DIR}/d_curvature_midbody_by_track.csv")
        p_log.f_exit(f_id)

    def run(self):
        f_id = p_log.f_enter("TracePipeline::run")

        self.analyze_all_frames()
        self.identify_worms()
        self.save_outputs()
        
        p_log.f_exit(f_id)


## Testing

In [382]:
def test_preprocessing(input_video, output_dir):
    video_loader = VideoLoader(input_video)
    background = find_background(video_loader, n=100, k=20)
    video_loader.reset()
    frame = video_loader.get_next_frame()
    frame_dat = preprocess_frame(frame, background)
    cv2.imwrite(f"{output_dir}/raw_frame.png", frame)
    cv2.imwrite(f"{output_dir}/preprocessed_frame.png", frame_dat["mask"])

In [383]:
OUTPUT_DIR = '../output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def test(input_dir):
    ALL_DATA = get_all_output_paths(input_dir)
    video_path = ALL_DATA["video"]
    
    test_preprocessing(video_path, OUTPUT_DIR)
    #pipeline = TracePipeline(video_path, OUTPUT_DIR)
    #pipeline.run()

test(video_directories[3])

print(p_log.function_log)

Entered Function: find_background
Exiting Function: find_background
	Time Elapsed: 0.9424625519895926


NameError: name 'tight_low' is not defined